# 나비효과 — 2018-2021 V2Themes 재수집 + V3 분류

## 목적
기존 2018-2021 데이터는 구버전 파이프라인으로 수집되어 **v2_themes가 누락**됨.  
BigQuery `gkg_partitioned` 테이블에서 URL 기준으로 V2Themes를 재수집하고,  
V3 theme mapping (`gdelt_theme_mapping.yaml`)으로 분류하여 기존 V3 파일에 병합.

## 예상 효과
- 2018-2021: 현재 1.6% → 목표 30-40% 분류율
- 전체: 현재 41.0% → 목표 55-65% 분류율

## BQ 비용
- 4년 × GKG 파티션 = ~300-400GB 스캔
- 무료 티어 1TB/월 이내

## 실행 순서
1. BQ 인증
2. URL 목록 로드 (risk_events_classified_v3에서 2018-2021 URL 추출)
3. 반기별 BQ 쿼리 → url + v2_themes 수집
4. V3 theme mapping 분류
5. 기존 V3 parquet에 병합

In [ ]:
# ═══════════════════════════════════════
# 0. 환경 설정
# ═══════════════════════════════════════
# Colab에서 실행 시 아래 주석 해제
# !pip install google-cloud-bigquery db-dtypes pyarrow pandas pyyaml

from google.colab import auth
auth.authenticate_user()
print('✓ Google Cloud 인증 완료')

In [ ]:
# ═══════════════════════════════════════
# 1. Config & Imports
# ═══════════════════════════════════════
import gc, math, time
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.cloud import bigquery

# ── Google Drive 마운트 ──
from google.colab import drive
drive.mount('/content/drive')

# ── 경로 설정 (본인 Drive 경로에 맞게 수정) ──
BASE = Path('/content/drive/MyDrive/nabi_hyoghaw')
V3_FILE = BASE / 'data' / 'processed' / 'risk_events_classified_v3.parquet'
THEME_MAP_FILE = BASE / 'configs' / 'gdelt_theme_mapping.yaml'
OUTPUT_LOOKUP = Path('/content/theme_lookup_2018_2021.parquet')

PROJECT_ID = 'nabi-effect-project'  # 본인 GCP 프로젝트 ID

# ── 분류 설정 (classify_v3_themes.py와 동일) ──
S_MAX = 30.0
MIN_W = 2
MAX_CATS = 4
RTYPES = ['geopolitics','logistics','natural','regulatory',
          'market','labor','technology','esg','pandemic']

print(f'V3 file: {V3_FILE}')
print(f'Exists: {V3_FILE.exists()}')

In [ ]:
# ═══════════════════════════════════════
# 2. Theme Mapping 로드 (gdelt_theme_mapping.yaml)
# ═══════════════════════════════════════
import yaml

with open(THEME_MAP_FILE) as f:
    cfg = yaml.safe_load(f)

tmap = {}
for rt in RTYPES:
    if rt in cfg and 'themes' in cfg[rt]:
        tmap[rt] = [(t['prefix'].upper(), t['weight']) for t in cfg[rt]['themes']]
    else:
        tmap[rt] = []

total_prefixes = sum(len(v) for v in tmap.values())
print(f'✓ Theme mapping 로드: {total_prefixes} prefixes across {len(RTYPES)} categories')
for rt in RTYPES:
    print(f'  {rt:15s}: {len(tmap[rt]):3d} prefixes')

In [ ]:
# ═══════════════════════════════════════
# 3. classify_themes 함수 (classify_v3_themes.py와 동일)
# ═══════════════════════════════════════
def classify_themes(themes_str, tmap):
    """GDELT v2_themes 문자열 → (risk_types, severity, conf) or None"""
    if not themes_str or pd.isna(themes_str):
        return None
    # theme code 추출: "ARMEDCONFLICT,123;SANCTIONS,456" → {"ARMEDCONFLICT", "SANCTIONS"}
    codes = set()
    for p in str(themes_str).split(';'):
        p = p.strip()
        if ',' in p:
            c = p.rsplit(',', 1)[0].strip().upper()
            if c and len(c) > 1:
                codes.add(c)
    if not codes:
        return None
    
    # prefix 매칭 → weight 합산
    cat_w = {}
    for rt, pfxs in tmap.items():
        w = 0
        for pfx, weight in pfxs:
            for c in codes:
                if c.startswith(pfx):
                    w += weight
                    break
        if w >= MIN_W:
            cat_w[rt] = w
    if not cat_w:
        return None
    
    # 상위 MAX_CATS 카테고리
    top = sorted(cat_w.items(), key=lambda x: -x[1])[:MAX_CATS]
    rt = ','.join(sorted(c for c, _ in top))
    tw = sum(w for _, w in top)
    sev = round(min(math.log(1+tw) / math.log(1+S_MAX), 1.0), 4)
    conf = round(min(0.3 + 0.1*len(top) + 0.01*len(codes), 1.0), 3)
    return (rt, sev, conf)

# 테스트
test = 'ARMEDCONFLICT,5;SANCTIONS,3;ENV_CLIMATECHANGE,2;ECON_STOCKMARKET,1'
result = classify_themes(test, tmap)
print(f'Test: {test}')
print(f'Result: {result}')

In [ ]:
# ═══════════════════════════════════════
# 4. 2018-2021 URL 목록 추출
# ═══════════════════════════════════════
# risk_events_classified_v3에서 2018-2021 + legacy classify_source인 URL만 추출
# → BQ에서 이 URL들의 v2_themes를 가져올 것

print('V3 파일에서 2018-2021 legacy URL 추출 중...')
t0 = time.time()

legacy_urls = set()
pf = pq.ParquetFile(V3_FILE)
for batch in pf.iter_batches(batch_size=100_000,
                              columns=['event_time', 'url', 'classify_source']):
    df = batch.to_pandas()
    mask = (df['classify_source'] == 'legacy') & (df['event_time'].dt.year.between(2018, 2021))
    urls = df.loc[mask, 'url'].dropna().unique()
    legacy_urls.update(urls)
    del df
    gc.collect()

print(f'✓ {len(legacy_urls):,} unique legacy URLs (2018-2021) [{time.time()-t0:.0f}s]')

In [ ]:
# ═══════════════════════════════════════
# 5. BigQuery에서 V2Themes 수집 (반기별 쿼리)
# ═══════════════════════════════════════
# 전략: URL 목록을 BQ에 올리지 않고, 전체 GKG에서 V2Themes를 가져온 뒤
# Python에서 URL 매칭. BQ 비용은 같지만 구현이 간단.
#
# 대안(더 효율적): URL을 BQ temp table에 올려서 JOIN
# → 아래 방법이 비용 초과 시 대안 사용

client = bigquery.Client(project=PROJECT_ID)

# 반기별 기간 정의
periods = [
    ('2018-01-01', '2018-07-01'),
    ('2018-07-01', '2019-01-01'),
    ('2019-01-01', '2019-07-01'),
    ('2019-07-01', '2020-01-01'),
    ('2020-01-01', '2020-07-01'),
    ('2020-07-01', '2021-01-01'),
    ('2021-01-01', '2021-07-01'),
    ('2021-07-01', '2022-01-01'),
]

LOOKUP_SCHEMA = pa.schema([
    ('url', pa.string()),
    ('risk_types_v3', pa.string()),
    ('severity_v3', pa.float64()),
    ('conf_v3', pa.float64()),
])

writer = pq.ParquetWriter(str(OUTPUT_LOOKUP), LOOKUP_SCHEMA, compression='snappy')
total_matched = 0
total_classified = 0

for date_start, date_end in periods:
    t0 = time.time()
    print(f'\n--- {date_start} ~ {date_end} ---')
    
    # BQ 쿼리: DocumentIdentifier + V2Themes만 가져옴 (최소 비용)
    query = f"""
    SELECT
        DocumentIdentifier AS url,
        V2Themes AS v2_themes
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE
        _PARTITIONTIME >= TIMESTAMP('{date_start}')
        AND _PARTITIONTIME < TIMESTAMP('{date_end}')
        AND V2Themes IS NOT NULL
        AND DocumentIdentifier IS NOT NULL
    """
    
    # dry-run으로 비용 확인
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    dry = client.query(query, job_config=job_config)
    gb = dry.total_bytes_processed / 1024**3
    print(f'  Estimated: {gb:.1f} GB')
    
    # 실제 쿼리 실행
    job_config = bigquery.QueryJobConfig(use_query_cache=True)
    result = client.query(query, job_config=job_config)
    
    # 결과를 배치로 처리 (메모리 관리)
    period_matched = 0
    period_classified = 0
    buf_url, buf_rt, buf_sev, buf_conf = [], [], [], []
    
    for page in result.result(page_size=50000):
        for row in page:
            url = row.url
            if url not in legacy_urls:
                continue
            period_matched += 1
            
            res = classify_themes(row.v2_themes, tmap)
            if res is None:
                continue
            
            rt, sev, conf = res
            buf_url.append(url)
            buf_rt.append(rt)
            buf_sev.append(sev)
            buf_conf.append(conf)
            period_classified += 1
            
            if len(buf_url) >= 10_000:
                tbl = pa.table({
                    'url': pa.array(buf_url, type=pa.string()),
                    'risk_types_v3': pa.array(buf_rt, type=pa.string()),
                    'severity_v3': pa.array(buf_sev, type=pa.float64()),
                    'conf_v3': pa.array(buf_conf, type=pa.float64()),
                })
                writer.write_table(tbl)
                buf_url, buf_rt, buf_sev, buf_conf = [], [], [], []
                del tbl
    
    # 남은 버퍼 flush
    if buf_url:
        tbl = pa.table({
            'url': pa.array(buf_url, type=pa.string()),
            'risk_types_v3': pa.array(buf_rt, type=pa.string()),
            'severity_v3': pa.array(buf_sev, type=pa.float64()),
            'conf_v3': pa.array(buf_conf, type=pa.float64()),
        })
        writer.write_table(tbl)
        del tbl
        buf_url, buf_rt, buf_sev, buf_conf = [], [], [], []
    
    gc.collect()
    elapsed = time.time() - t0
    pct = period_classified / max(period_matched, 1) * 100
    print(f'  Matched: {period_matched:,} | Classified: {period_classified:,} ({pct:.1f}%) [{elapsed:.0f}s]')
    total_matched += period_matched
    total_classified += period_classified

writer.close()
gc.collect()

sz = OUTPUT_LOOKUP.stat().st_size / 1024**2
pct = total_classified / max(total_matched, 1) * 100
print(f'\n✓ 수집 완료')
print(f'  Total matched: {total_matched:,}')
print(f'  Total classified: {total_classified:,} ({pct:.1f}%)')
print(f'  Lookup file: {sz:.1f} MB')

In [ ]:
# ═══════════════════════════════════════
# 6. 기존 V3 파일에 병합 (DuckDB JOIN)
# ═══════════════════════════════════════
# 2018-2021 legacy 행들의 risk_types, severity, classify_source, classify_conf를 업데이트

# Colab에 duckdb 설치
!pip install -q duckdb
import duckdb

MERGED_OUTPUT = BASE / 'data' / 'processed' / 'risk_events_classified_v3.parquet'
TEMP_OUTPUT = Path('/content/risk_events_classified_v3_merged.parquet')

con = duckdb.connect('/content/merge.duckdb')
con.execute('SET threads=2')
con.execute("SET memory_limit='4GB'")
con.execute("SET temp_directory='/content/tmp'")

# 핵심 쿼리: legacy 행만 lookup과 JOIN하여 업데이트
merge_query = f"""
SELECT
    e.event_id, e.event_time, e.url, e.title,
    -- risk_types: legacy 행에 대해 lookup 결과로 교체
    CASE
        WHEN e.classify_source = 'legacy'
             AND EXTRACT(YEAR FROM e.event_time) BETWEEN 2018 AND 2021
             AND l.risk_types_v3 IS NOT NULL
        THEN l.risk_types_v3
        ELSE e.risk_types
    END AS risk_types,
    -- severity
    CASE
        WHEN e.classify_source = 'legacy'
             AND EXTRACT(YEAR FROM e.event_time) BETWEEN 2018 AND 2021
             AND l.severity_v3 IS NOT NULL
        THEN l.severity_v3
        ELSE e.severity
    END AS severity,
    e.element, e.alias, e.company_id, e.score, e.finbert,
    e.country_ids, e.tone, e.source_domain, e.collection_type,
    -- classify_source 업데이트
    CASE
        WHEN e.classify_source = 'legacy'
             AND EXTRACT(YEAR FROM e.event_time) BETWEEN 2018 AND 2021
             AND l.risk_types_v3 IS NOT NULL
        THEN 'gdelt_themes'
        ELSE e.classify_source
    END AS classify_source,
    -- classify_conf 업데이트
    CASE
        WHEN e.classify_source = 'legacy'
             AND EXTRACT(YEAR FROM e.event_time) BETWEEN 2018 AND 2021
             AND l.conf_v3 IS NOT NULL
        THEN l.conf_v3
        ELSE e.classify_conf
    END AS classify_conf
FROM read_parquet('{V3_FILE}') e
LEFT JOIN (
    SELECT url, FIRST(risk_types_v3) AS risk_types_v3,
           FIRST(severity_v3) AS severity_v3, FIRST(conf_v3) AS conf_v3
    FROM read_parquet('{OUTPUT_LOOKUP}')
    GROUP BY url
) l ON e.url = l.url
"""

t0 = time.time()
print('Merging...')
con.execute(f"COPY ({merge_query}) TO '{TEMP_OUTPUT}' (FORMAT PARQUET, COMPRESSION SNAPPY, ROW_GROUP_SIZE 100000)")
elapsed = time.time() - t0
sz = TEMP_OUTPUT.stat().st_size / 1024**2
print(f'✓ Merge 완료: {sz:.0f} MB [{elapsed:.0f}s]')

con.close()

In [ ]:
# ═══════════════════════════════════════
# 7. 결과 통계 확인
# ═══════════════════════════════════════
con = duckdb.connect()

o = str(TEMP_OUTPUT)

# 전체 통계
total = con.execute(f"SELECT count(*) FROM read_parquet('{o}')").fetchone()[0]
other = con.execute(f"SELECT count(*) FROM read_parquet('{o}') WHERE risk_types='other'").fetchone()[0]
cls = total - other
print('=' * 60)
print('V3 + 2018-2021 병합 결과')
print('=' * 60)
print(f'  Total:      {total:,}')
print(f'  Classified: {cls:,} ({cls/total*100:.1f}%)')
print(f'  Other:      {other:,} ({other/total*100:.1f}%)')

# 연도별
print('\n  연도별:')
r = con.execute(f"""
SELECT EXTRACT(YEAR FROM event_time) as yr, count(*) as n,
  sum(CASE WHEN risk_types!='other' THEN 1 ELSE 0 END) as classified
FROM read_parquet('{o}')
GROUP BY 1 ORDER BY 1
""").fetchdf()
for _, row in r.iterrows():
    yr = row['yr']; n = int(row['n']); c = int(row['classified'])
    pct = c / max(n, 1) * 100
    print(f'    {yr}: {n:>10,} | classified {c:,} ({pct:.1f}%)')

# 카테고리별
print('\n  카테고리별:')
for rt in ['geopolitics','logistics','natural','regulatory','market','labor','technology','esg','pandemic']:
    n = con.execute(f"SELECT count(*) FROM read_parquet('{o}') WHERE risk_types LIKE '%{rt}%'").fetchone()[0]
    print(f'    {rt:15s}: {n:>10,} ({n/total*100:.2f}%)')

# classify_source별
print('\n  By classify_source:')
r2 = con.execute(f"""
SELECT classify_source, count(*) n,
  sum(CASE WHEN risk_types!='other' THEN 1 ELSE 0 END) hit,
  round(avg(classify_conf), 3) c
FROM read_parquet('{o}') GROUP BY 1 ORDER BY n DESC
""").fetchdf()
for _, row in r2.iterrows():
    s = row['classify_source']; n = int(row['n']); h = int(row['hit']); c = row['c']
    print(f'    {s:20s}: {n:>10,} | classified {h:,} ({h/max(n,1)*100:.1f}%) | conf={c:.3f}')

con.close()

In [ ]:
# ═══════════════════════════════════════
# 8. 최종 파일 저장 (Drive에 복사)
# ═══════════════════════════════════════
# 통계 확인 후 문제없으면 실행

import shutil

# 기존 파일 백업
backup = V3_FILE.parent / 'risk_events_classified_v3_before_2018_merge.parquet'
if V3_FILE.exists() and not backup.exists():
    shutil.copy2(str(V3_FILE), str(backup))
    print(f'✓ 기존 파일 백업: {backup.name}')

# 병합 파일을 Drive로 복사
shutil.copy2(str(TEMP_OUTPUT), str(V3_FILE))
sz = V3_FILE.stat().st_size / 1024**2
print(f'✓ 최종 파일 저장: {V3_FILE.name} ({sz:.0f} MB)')
print(f'\n다음 단계: 로컬에서 Stage 6-10 파이프라인 실행')

## 트러블슈팅

### BQ 비용이 무료 티어 초과 시
1. `periods` 리스트에서 일부 기간만 실행 (예: 2020-2021만 먼저)
2. 나머지 기간은 다음 달 무료 티어 리셋 후 실행

### 메모리 부족 시 (Colab 런타임 크래시)
1. `page_size`를 50000 → 10000으로 줄임
2. 기간을 분기별로 더 세분화
3. GPU 런타임 사용 (메모리 더 많음)

### URL 매칭률이 낮을 경우
- GDELT DocumentIdentifier = full URL (http/https 포함)
- risk_events_final_v2의 url 형식과 동일한지 확인
- 필요시 URL 정규화 (trailing slash, www 제거 등)